# 01 · From a language model to an application

**Session 01 · Building AI applications for research and data · notebook 1 of 6**

## Objective

See what a model does with a research question on its own, what changes when it is given instructions and context, and why instructions and context are the first two components of an application.

## The application after this notebook

Diagram: the question and the context feed the prompt, which goes to the model and produces the answer.

The prompt is the one piece this notebook builds; the rest of the application is added in the notebooks that follow.

The worked example for the whole session is one question:

> How have internet access and economic development evolved in Rwanda and Kenya? Compare the recent available data, summarize the main patterns, and cite the sources.

## What you will see

Four short scenes, each with a result you can read:

1. The model answers from what it learned in training. The answer sounds right, and nothing in it can be checked.
2. The same question with four country briefs supplied. The answer is specific and cites its sources.
3. A question the briefs cannot answer, and what the model does about it.
4. A question with two meanings, and what changes when the instructions tell the model to ask.

## How to run it

Run the cells in order. Without an API key the model outputs are recorded ones, labelled `[recorded output: ...]`; each cell names its scenario with `w.use(...)` before calling the model. With a key in `WORKSHOP_API_KEY` the same cells call the model live.

The next cell installs the session's tools and loads its helper module. It takes about a minute the first time, and you do not need to read it.

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/avsolatorio/ai4data.git"   # replace with the published repository when released
BRANCH = "feat/training-series"
SESSION = "training/session-01-building-ai-applications/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook replays recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
model = w.make_model()
print("Model:", w.mode(model))

Model: recorded outputs (no WORKSHOP_API_KEY set)


In [2]:
#@title Helpers that draw the table and the prompt below { display-mode: "form" }
from html import escape
from IPython.display import HTML, display


def claim_check_html(rows, source="none"):
    """Each row is (claim, what the model said, what the data says, verdict). Verdict "close" is green."""
    colours = {"close": ("#dff3e7", "#1f6b43"), "none": ("#fbe0dd", "#8e2b21")}
    cell = 'style="padding:8px 10px;border-bottom:1px solid #e5e9f0;vertical-align:top"'

    def pill(text):
        bg, fg = colours.get(text, ("#fdebd0", "#8a4b0a"))
        return f'<span style="display:inline-block;padding:2px 10px;border-radius:999px;background:{bg};color:{fg};font-weight:600;font-size:12.5px">{escape(text)}</span>'

    body = "".join(
        f'<tr><td {cell}><b>{escape(claim)}</b></td><td {cell}>{escape(said)}</td><td {cell}>{escape(data)}</td>'
        f'<td {cell}>{pill(verdict)}</td><td {cell}>{pill(source)}</td></tr>'
        for claim, said, data, verdict in rows
    )
    close = sum(1 for row in rows if row[3] == "close")
    summary = f"{close} of {len(rows)} claims land close to the data."
    if source == "none":
        summary += " The others are marked above. None names a source, so a reader cannot tell which is which."
    head = "".join(f'<th style="padding:8px 10px">{h}</th>' for h in ["Claim", "What the model said", "What the data says", "Verdict", "Source named"])
    return (
        f'<table style="border-collapse:collapse;font:14px Helvetica,Arial,sans-serif;width:100%">'
        f'<thead><tr style="text-align:left;background:#f3f4f6">{head}</tr></thead><tbody>{body}</tbody></table>'
        f'<p style="font:600 14px Helvetica,Arial,sans-serif;margin:10px 2px">{escape(summary)}</p>'
    )

def show_claim_check(rows, source="none"):
    display(HTML(claim_check_html(rows, source)))

def prompt_html(messages):
    """Messages as produced by prompt.format_messages: a system message, then a human message of the form
    "CONTEXT:\\n...\\n\\nQUESTION: ..."."""
    colours = {"instructions": "#2f5f9e", "context": "#2f8f5b", "question": "#d9822b"}

    def block(kind, meta, text):
        c = colours[kind]
        return (f'<div style="border-left:5px solid {c};background:{c}1a;padding:8px 12px;margin:8px 0;border-radius:0 8px 8px 0">'
                f'<div style="font:700 12.5px Helvetica,Arial,sans-serif;color:{c};margin-bottom:4px">{kind.capitalize()} '
                f'<span style="font-weight:400;color:#6b7280">· {escape(meta)}</span></div>'
                f'<pre style="margin:0;white-space:pre-wrap;font:12.5px Menlo,Consolas,monospace;color:#16315a">{escape(text)}</pre></div>')

    system = next(m for m in messages if m.type == "system").content
    human = next(m for m in messages if m.type == "human").content
    head, _, question = human.rpartition("\n\nQUESTION: ")
    context = head[len("CONTEXT:\n"):] if head.startswith("CONTEXT:\n") else head
    shown = context if len(context) <= 330 else context[:330].rsplit(" ", 1)[0] + " …"
    return (block("instructions", f"the system message, {len(system):,} characters", system)
            + block("context", f"{len(context):,} characters" + ("" if shown == context else ", the first lines shown"), shown)
            + block("question", "from the user", question))

def show_prompt(messages):
    display(HTML(prompt_html(messages)))

## The stack

| Need | Library |
|---|---|
| Documents, splitting, embeddings, vector store, prompts, tools, structured output | [LangChain](https://python.langchain.com/) |
| The agent loop | [LangGraph](https://langchain-ai.github.io/langgraph/), through `langchain.agents.create_agent` |
| Tool server and client over the Model Context Protocol | [FastMCP](https://gofastmcp.com/) |
| Number verification | [`pcn-core`](https://github.com/worldbank/pcn), the Python core of Proof-Carrying Numbers |
| Output schemas | [Pydantic](https://docs.pydantic.dev/) |

`workshop.py` holds what these do not: the worked-example data, a chat model that replays recorded outputs when no API key is configured, and a few printing and evaluation helpers.

## 1. A prompt with three parts

Three parts go into every call: **instructions** (the system message), **context** (evidence supplied to the model), and the **question**. LangChain's `ChatPromptTemplate` keeps them as separate slots, and `prompt | model` is the whole application for now. Keeping the parts separate is the first design decision, because each is changed by a different person for a different reason.

Animation: instructions set by the developer, context supplied by the application and a question asked by the user are combined into one prompt, which goes to the model and produces an answer that cites its source.

*The animation loops. The text in each box is taken from the cells below.*

In [3]:
from langchain_core.prompts import ChatPromptTemplate

INSTRUCTIONS_BASIC = "You are a research assistant. Answer the question."

INSTRUCTIONS_GROUNDED = """You are a research assistant for development data.
Answer only from the supplied context. Cite the document id in square brackets after each figure.
If the context does not contain what is needed, say so and give the closest available information."""

INSTRUCTIONS_CLARIFY = INSTRUCTIONS_GROUNDED + """
If the question is ambiguous, say which readings are possible and ask which one is meant before answering."""

prompt = ChatPromptTemplate.from_messages([
    ("system", "{instructions}"),
    ("human", "CONTEXT:\n{context}\n\nQUESTION: {question}"),
])
chain = prompt | model

print(w.MAIN_QUESTION)

How have internet access and economic development evolved in Rwanda and Kenya? Compare the recent available data, summarize the main patterns, and cite the sources.


## 2. Zero context: the model answers from its training data

The baseline. No evidence is supplied, so the model answers from what it absorbed in training.

In [4]:
w.use("nb01_zero_context")
answer = chain.invoke({"instructions": INSTRUCTIONS_BASIC, "context": "(none)", "question": w.MAIN_QUESTION})
w.show(answer)

[recorded output: nb01_zero_context]
Based on what I know up to my training data, both countries have expanded internet access and grown
their economies, though from different starting points.

Kenya is one of East Africa's most connected economies. Internet penetration is often reported above
80 percent of the population when measured by subscriptions, and mobile money through M-Pesa is
widespread. GDP per capita is roughly 2,000 US dollars, and growth has averaged around 5 percent a
year over the past decade.

Rwanda has invested heavily in broadband infrastructure and e-government. Internet penetration is
commonly cited at around 30 percent of the population, and GDP per capita is close to 1,000 US
dollars, with growth frequently above 7 percent.

Please note that these figures are approximate and may be out of date. For precise, current values
you should consult the World Bank's World Development Indicators or the ITU.


Read the answer against the data. The first cell prints the latest values from the World Development Indicators that the session uses as its reference. The second sets six claims from the answer beside them.

In [5]:
for ind in ["IT.NET.USER.ZS", "NY.GDP.PCAP.CD"]:
    for country in ["Kenya", "Rwanda"]:
        r = w.indicator_lookup(ind, country, start=2023)
        print(f"{r['name']:50s} {country:7s}", [(o["year"], round(o["value"], 1)) for o in r["observations"]])

Individuals using the Internet (% of population)   Kenya   [(2023, 32.1), (2024, 35.0)]
Individuals using the Internet (% of population)   Rwanda  [(2023, 28.8), (2024, 31.7)]
GDP per capita (current US$)                       Kenya   [(2023, 1942.6), (2024, 2133.5)]
GDP per capita (current US$)                       Rwanda  [(2023, 1061.1), (2024, 1059.9)]


In [6]:
def value(ind, country):
    return w.latest(w.indicator_lookup(ind, country, start=2023))["value"]


def growth(country):
    return [o["value"] for o in w.indicator_lookup("NY.GDP.MKTP.KD.ZG", country, start=2015)["observations"]]


rows = [  # (claim, what the model said, what the data says, verdict)
    ("Kenya · internet use", "“above 80 percent… when measured by subscriptions”",
     f"{value('IT.NET.USER.ZS', 'Kenya'):.1f}% of people online; {value('IT.CEL.SETS.P2', 'Kenya'):.1f} mobile subscriptions per 100", "blends two measures"),
    ("Kenya · GDP per capita", "“roughly 2,000 US dollars”", f"{value('NY.GDP.PCAP.CD', 'Kenya'):,.0f} US dollars", "close"),
    ("Kenya · growth", "“around 5 percent a year over the past decade”", f"{sum(growth('Kenya')) / len(growth('Kenya')):.1f} percent a year, 2015 to 2024", "close"),
    ("Rwanda · internet use", "“around 30 percent”", f"{value('IT.NET.USER.ZS', 'Rwanda'):.1f}% of people online", "close"),
    ("Rwanda · GDP per capita", "“close to 1,000 US dollars”", f"{value('NY.GDP.PCAP.CD', 'Rwanda'):,.0f} US dollars", "close"),
    ("Rwanda · growth", "“frequently above 7 percent”", f"above 7 percent in {sum(g > 7 for g in growth('Rwanda'))} of {len(growth('Rwanda'))} years, 2015 to 2024", "close"),
]
show_claim_check(rows)

Claim,What the model said,What the data says,Verdict,Source named
Kenya · internet use,“above 80 percent… when measured by subscriptions”,35.0% of people online; 126.5 mobile subscriptions per 100,blends two measures,none
Kenya · GDP per capita,"“roughly 2,000 US dollars”","2,133 US dollars",close,none
Kenya · growth,“around 5 percent a year over the past decade”,"4.6 percent a year, 2015 to 2024",close,none
Rwanda · internet use,“around 30 percent”,31.7% of people online,close,none
Rwanda · GDP per capita,"“close to 1,000 US dollars”","1,060 US dollars",close,none
Rwanda · growth,“frequently above 7 percent”,"above 7 percent in 7 of 10 years, 2015 to 2024",close,none


What to notice:

- Five of the six claims are close to the data, so the answer sounds reliable.
- One sentence blends two measures. "Above 80 percent" fits mobile subscriptions (126.5 per 100 people in 2024), not the share of people online (35.0 percent), and nothing in the answer tells the reader which one is meant.
- No claim names a source or a year, so a reader cannot tell the sound figures from the blended one. The model hedges, which is the right behaviour for a model with no evidence, and of no use to a reader who needs a number they can check.

Prompting changes the style of this answer. It cannot give the model data it does not have.

## 3. The same question with context

The four country briefs from the session's document collection are supplied as context. The instructions now require citations and allow the model to say when something is missing.

In [7]:
docs = {d.metadata["id"]: d for d in w.load_documents()}
context = "\n\n".join(f"[{i}]\n{docs[i].page_content}" for i in ["kenya-digital", "rwanda-digital", "kenya-economy", "rwanda-economy"])
print(f"{len(context):,} characters of context, roughly {len(context)//4:,} tokens\n")

w.use("nb01_with_context")
answer = chain.invoke({"instructions": INSTRUCTIONS_GROUNDED, "context": context, "question": w.MAIN_QUESTION})
w.show(answer)

4,076 characters of context, roughly 1,019 tokens

[recorded output: nb01_with_context]
Internet use. In Kenya, the share of individuals using the internet rose from 7.2 percent of the
population in 2010 to 35.0 percent in 2024 [kenya-digital]. In Rwanda it rose from 8.0 percent in
2010 to 31.7 percent in 2024 [rwanda-digital]. Kenya's fastest growth came between 2010 and 2015;
Rwanda's came between 2015 and 2020. Both countries ended the period with about a third of the
population online.

Economic development. Kenya's GDP per capita in current US dollars rose from 1,092 in 2010 to 2,133
in 2024, with a dip to 1,943 in 2023 that coincided with a currency depreciation [kenya-economy].
Rwanda's rose from 587 in 2010 to 1,060 in 2024, passing 1,000 in 2022 [rwanda-economy]. Rwanda grew
faster in most years, above 7 percent except in 2020, while Kenya grew in every year except 2020.

Patterns. Kenya has the higher level on both indicators; Rwanda has the faster growth on both.
Internet us

Diagram: the same question without context gets an answer with no source, no date and a blend of two measures; with the four country briefs as context it gets a dated answer that cites its source.

*The two answers, shortened. They are the recorded outputs of sections 2 and 3.*

The figures now come from the briefs, each one is cited, and the comparison is specific. The model did none of the finding; the application did, by putting the right text in front of it.

## 4. Handling missing information

An application has to behave well when the evidence does not contain the answer. The instruction "say so and give the closest available information" is what makes the difference.

In [8]:
w.use("nb01_missing_info")
answer = chain.invoke({"instructions": INSTRUCTIONS_GROUNDED, "context": docs["kenya-digital"].page_content,
                       "question": "What share of Kenya's population used the internet in 2025?"})
w.show(answer)

[recorded output: nb01_missing_info]
The supplied material contains no observation for 2025. The latest available year for the share of
Kenya's population using the internet is 2024, at 35.0 percent [kenya-digital]. If a 2025 value is
needed, it will have to come from a later release of the World Development Indicators or from the
ITU.


## 5. The ambiguous question under two sets of instructions

"How connected is Kenya?" can mean internet use, mobile subscriptions, broadband, or something else. Compare what the model does with the basic instructions and with an instruction to ask.

In [9]:
print("Without an instruction to clarify:\n")
w.use("nb01_ambiguous_no_instruction")
w.show(chain.invoke({"instructions": INSTRUCTIONS_BASIC, "context": docs["kenya-digital"].page_content, "question": "How connected is Kenya?"}))
print("\n" + "=" * 100 + "\nWith the instruction:\n")
w.use("nb01_ambiguous_with_instruction")
w.show(chain.invoke({"instructions": INSTRUCTIONS_CLARIFY, "context": docs["kenya-digital"].page_content, "question": "How connected is Kenya?"}))

Without an instruction to clarify:

[recorded output: nb01_ambiguous_no_instruction]
Kenya is highly connected. Mobile cellular subscriptions reached 126.5 per 100 people in 2024, which
is more than one subscription per person, and 35.0 percent of the population used the internet in
the same year. Kenya is also the home of M-Pesa, so mobile money use is very high. Overall, Kenya is
one of the most connected countries in East Africa.

With the instruction:

[recorded output: nb01_ambiguous_with_instruction]
"Connected" can mean several things, and the supplied material measures two of them. Which do you
mean?

- Share of individuals using the internet: 35.0 percent of the population in 2024 [kenya-digital].
- Mobile cellular subscriptions: 126.5 per 100 people in 2024, a count of subscriptions that can
exceed the number of people [kenya-digital].

The two measures answer different questions. Subscriptions passed one per person around 2020, while
internet use stood at about a third of th

The first answer picks a reading and runs with it. The second names the readings and asks. For a research application the second is the behaviour to specify, because a confident answer to the wrong question looks the same as a confident answer to the right one.

## 6. What the model received

The messages the prompt produced for the last call, with each part in the colour it had in section 1: the instructions in the system message, then the context and the question in the human message.

In [10]:
messages = prompt.format_messages(instructions=INSTRUCTIONS_CLARIFY, context=docs["kenya-digital"].page_content, question="How connected is Kenya?")
show_prompt(messages)

## 7. What context costs

Four short briefs took about a thousand tokens. A real collection has thousands of documents and cannot be pasted into every call: the context window has a limit, cost grows with every token, and the relevant passages are a small part of the whole. The next two notebooks prepare a collection so that the application can select what to supply.

Diagram: the four briefs of this notebook, about 1,019 tokens, fit inside the model's context window, while a real collection of thousands of documents cannot, so the application must choose what the model reads.

## Key points

- An application separates instructions, context, and the question. Each is changed by a different person for a different reason.
- Instructions shape behaviour: citations, what to do when information is missing, when to ask. They cannot supply data the model does not have.
- Supplied context is what makes the answer specific and checkable. Selecting it is the job of the rest of the application.

## Exercise

1. Read the answers of sections 2 and 3 side by side. For each number, note whether you could check it and where. Then say which answer you would send to a colleague, and why.
2. Change `INSTRUCTIONS_GROUNDED` so that the answer is a table with one row per country and columns for the indicator, the latest year, the value, and the source. Rerun section 3. With a live key the output changes; in recorded mode the cell returns the recorded answer, so note the expected result instead.

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows.
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.